In [ ]:
import random
import os

import faiss
import pickle
import numpy as np
from sklearn.decomposition import PCA

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

%cd ..

/


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.8 MB/s eta 0:00:00


In [ ]:
def get_memory(index):
    # write index to file
    faiss.write_index(index, './temp.index')
    # get file size
    file_size = os.path.getsize('./temp.index')
    # delete saved index
    os.remove('./temp.index')
    print(f"File size: {file_size/1024**2} MB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Setting up the data

In [ ]:
feature_list = pickle.load(
    open('/content/drive/MyDrive/features/features-caltech101-resnet.pickle', 'rb')
)
# feature_list = feature_list/np.linalg.norm(feature_list,axis=1).reshape(-1,1)  # normalize features so each column has length 1
pca = PCA(n_components=128)
feature_list_compressed = pca.fit_transform(feature_list)


# Flat (Ground Truth)

In [ ]:
index = faiss.IndexFlatL2(2048)
index.train(feature_list)
index.add(feature_list)

index_compressed = faiss.IndexFlatL2(128)
index_compressed.train(feature_list_compressed)
index_compressed.add(feature_list_compressed)

get_memory(index)
get_memory(index_compressed)

File size: 71.43754291534424 MB
File size: 4.464886665344238 MB


# IVFFlat

In [ ]:
index_ivf = faiss.index_factory(2048, 'IVF100,Flat')
index_ivf.train(feature_list)
index_ivf.add(feature_list)

index_ivf_compressed = faiss.index_factory(128, 'IVF100,Flat')
index_ivf_compressed.train(feature_list_compressed)
index_ivf_compressed.add(feature_list_compressed)


get_memory(index_ivf)
get_memory(index_ivf_compressed)

File size: 72.28940868377686 MB
File size: 4.5843305587768555 MB


# IVFPQ
- 30x memory savings compared to IVFFlat

In [ ]:
nlist = 100
m = 8
nbits=8

quantizer = faiss.IndexFlatL2(2048)  # this remains the same
index_ivfpq = faiss.IndexIVFPQ(quantizer, 2048, nlist, m, nbits)
# index_ivfpq = faiss.index_factory(2048, 'IVF100,PQ8')
index_ivfpq.train(feature_list)
index_ivfpq.add(feature_list)

quantizer = faiss.IndexFlatL2(128)  # this remains the same
index_ivfpq_compressed = faiss.IndexIVFPQ(quantizer, 128, nlist, m, nbits)
# index_ivfpq_compressed = faiss.index_factory(128, 'IVF100,PQ8')
index_ivfpq_compressed.train(feature_list_compressed)
index_ivfpq_compressed.add(feature_list_compressed)

get_memory(index_ivfpq)
get_memory(index_ivfpq_compressed)

File size: 2.921710968017578 MB
File size: 0.3142890930175781 MB


# OPQIVFPQ
- Extremely slow for large dimensions  (https://github.com/facebookresearch/faiss/issues/625)
- Interrupted kernel at 4 mins for 2048 dimensions, only attempting 128 dimensions for OPQ
- Slightly larger index than IVFPQ

In [ ]:
index_opqivfpq_compressed = faiss.index_factory(128, 'OPQ8,IVF100,PQ8')
index_opqivfpq_compressed.train(feature_list_compressed)
index_opqivfpq_compressed.add(feature_list_compressed)

get_memory(index_opqivfpq_compressed)

File size: 0.37685680389404297 MB


In [ ]:
random_image_indices = random.sample(range(0, len(feature_list)), 100)
random_feature_list = np.array([feature_list[each_index] for each_index in random_image_indices])
random_feature_list_compressed = np.array([feature_list_compressed[each_index] for each_index in random_image_indices])

# Ground truth distances

In [ ]:
D_flat, I_flat = index.search(random_feature_list, 6)
D_flat_compressed, I_flat_compressed = index_compressed.search(random_feature_list_compressed, 5)

# Comparing indexes against ground truth
- IVF 100 is very high, close to known number of clusters at ~101, which helps recall
- IVFFlat is almost same size as IVFFlat (compressed) but slightly higher recall
- Strangely IVFPQ compressed from 2048 to 128 dimensions increases recall@5 0.44 to 0.66
- When trying to improve IVFPQ, lowering nbits from 8 to 7 to address the warnings leaves exact same 0.44 recall on IVFPQ and worsens IVFPQ compressed from 0.66 to 0.6 -> may be good to ignore warnings
- For this dataset, IVFPQ has 30x smaller memory footprint than IVFFlat but only slightly lower recall 0.66 vs 0.79

In [ ]:
def compare(I_gt,feature_list,index, name=''):
    D, I = index.search(feature_list, 6)
    print(f'Recall@5 for {name}: {(I[:,1:6] == I_gt[:,[1]]).sum()/len(feature_list)}')

In [ ]:
index_names = ['IVFFlat','IVFPQ']
index_names_compressed = ['IVFFlat (compressed)','IVFPQ (compressed)','OPQIVFPQ (compressed)']

indexes = [index_ivf,
           index_ivfpq]

indexes_compressed = [index_ivf_compressed,
                      index_ivfpq_compressed,
                      index_opqivfpq_compressed]

for name, idx in zip(index_names, indexes):
    compare(I_flat,random_feature_list,idx, name)


for name, idx in zip(index_names_compressed, indexes_compressed):
    compare(I_flat_compressed,random_feature_list_compressed,idx, name)

Recall@5 for IVFFlat: 0.79
Recall@5 for IVFPQ: 0.45
Recall@5 for IVFFlat (compressed): 0.84
Recall@5 for IVFPQ (compressed): 0.67
Recall@5 for OPQIVFPQ (compressed): 0.71


## Checking what lowers recall

In [ ]:
index_ivf_compressed = faiss.index_factory(128, 'IVF100,PQ8')
index_ivf_compressed.train(feature_list_compressed)
index_ivf_compressed.add(feature_list_compressed)

compare(I_flat_compressed,random_feature_list_compressed,index_ivf_compressed)

Recall@5 for : 0.67


## Experiments
- Lower nlist 100 to 50
- Lower PQ M 8 to 2

Results
- nlist of IVF has much larger impact on recall@5 than nbits of PQ


In [ ]:
index_ivfpq_compressed = faiss.index_factory(128, 'IVF100,PQ2')
index_ivfpq_compressed.train(feature_list_compressed)
index_ivfpq_compressed.add(feature_list_compressed)

compare(I_flat_compressed,random_feature_list_compressed,index_ivfpq_compressed)

index_ivfpq_compressed = faiss.index_factory(128, 'IVF50,PQ8')
index_ivfpq_compressed.train(feature_list_compressed)
index_ivfpq_compressed.add(feature_list_compressed)

compare(I_flat_compressed,random_feature_list_compressed,index_ivfpq_compressed)

index_ivfpq_compressed = faiss.index_factory(128, 'IVF50,PQ2')
index_ivfpq_compressed.train(feature_list_compressed)
index_ivfpq_compressed.add(feature_list_compressed)

compare(I_flat_compressed,random_feature_list_compressed,index_ivfpq_compressed)

Recall@5 for : 0.46
Recall@5 for : 0.63
Recall@5 for : 0.3


# Custom IndexPQ

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances

class CustomIndexPQ:

    BITS2DTYPE = {
        8: np.uint8,
        16: np.uint16,
    }

    def __init__(self,d: int,m: int,nbits: int,) -> None:
        """Custom IndexPQ implementation.
        Parameters
        ----------
        d
            Dimensionality of the original vectors.
        m
            Number of segments.
        nbits
            Number of bits.
        """
        if d % m != 0:
            raise ValueError("d needs to be a multiple of m")

        if nbits not in CustomIndexPQ.BITS2DTYPE:
            raise ValueError(f"Unsupported number of bits {nbits}")

        self.m = m
        self.k = 2**nbits
        self.d = d
        self.ds = d // m

        self.estimators = [KMeans(n_clusters=self.k, random_state=1) for _ in range(m)]

        self.is_trained = False

        self.dtype = CustomIndexPQ.BITS2DTYPE[nbits]
        self.dtype_orig = np.float32

    def train(self, X: np.ndarray) -> None:
        """Train M KMeans estimators.
        Parameters
        ----------
        X
            Array of shape `(n, d)` and dtype `float32`.
        """
        for i in range(self.m):
            estimator = self.estimators[i]
            X_i = X[:, i * self.ds : (i + 1) * self.ds]
            estimator.fit(X_i)

        self.is_trained = True


    def encode(self, X: np.ndarray) -> np.ndarray:
        """Encode original features into codes.
        Parameters
        ----------
        X
            Array of shape `(n_queries, d)` of dtype `np.float32`.
        Returns
        -------
        result
            Array of shape `(n_queries, m)` of dtype `np.uint8`.
        """

        n = len(X)
        result = np.empty((n, self.m), dtype=self.dtype)  #Prevents automatic 'float64' causing IndexError: arrays used as indices must be of integer (or boolean) type

        for i in range(self.m):
            estimator = self.estimators[i]
            X_i = X[:, i * self.ds : (i + 1) * self.ds]
            result[:, i] = estimator.predict(X_i)

        return result

    def add(self, X: np.ndarray) -> None:
        """Add vectors to the database (their encoded versions).
        Parameters
        ----------
        X
            Array of shape `(n_codes, d)` of dtype `np.float32`.
        """
        if not self.is_trained:
            raise ValueError("The quantizer needs to be trained first.")
        self.codes = self.encode(X)

    def compute_asymmetric_distances(self, X: np.ndarray) -> np.ndarray:
        """Compute asymmetric distances to all database codes.
        Parameters
        ----------
        X
            Array of shape `(n_queries, d)` of dtype `np.float32`.
        Returns
        -------
        distances
            Array of shape `(n_queries, n_codes)` of dtype `np.float32`.
        """
        if not self.is_trained:
            raise ValueError("The quantizer needs to be trained first.")

        if self.codes is None:
            raise ValueError("No codes detected. You need to run `add` first")

        n_queries = len(X)
        n_codes = len(self.codes)

        distance_table = np.empty(
            (n_queries, self.m, self.k), dtype=self.dtype_orig
        )  # (n_queries, m, k)

        for i in range(self.m):
            X_i = X[:, i * self.ds : (i + 1) * self.ds]  # (n_queries, ds)
            centers = self.estimators[i].cluster_centers_  # (k, ds)
            distance_table[:, i, :] = euclidean_distances(X_i,
                                                          centers,
                                                          squared=True)

        distances = np.zeros((n_queries, n_codes), dtype=self.dtype_orig)

        for i in range(self.m):
            distances += distance_table[:, i, self.codes[:, i]]

        return distances

    def search(self, X: np.ndarray, k: int) -> tuple:
        """Find k closest database codes to given queries.
        Parameters
        ----------
        X
            Array of shape `(n_queries, d)` of dtype `np.float32`.
        k
            The number of closest codes to look for.
        Returns
        -------
        distances
            Array of shape `(n_queries, k)`.
        indices
            Array of shape `(n_queries, k)`.
        """
        n_queries = len(X)
        distances_all = self.compute_asymmetric_distances(X)

        indices = np.argsort(distances_all, axis=1)[:, :k]

        distances = np.empty((n_queries, k), dtype=np.float32)
        for i in range(n_queries):
            distances[i] = distances_all[i][indices[i]]

        return distances, indices

In [ ]:
d, m, nbits = 128,8,8
custom = CustomIndexPQ(d, m, nbits)

In [ ]:
custom.train(feature_list_compressed)
custom.add(feature_list_compressed)

In [ ]:
D_custom, I_custom = custom.search(random_feature_list_compressed, 5)
D_custom[:5], I_custom[:5]

(array([[10325.34 , 18945.953, 19623.104, 20674.9  , 20699.363],
        [11861.067, 18370.176, 19944.418, 19945.201, 20197.004],
        [10590.117, 15088.973, 15371.737, 16104.916, 16130.048],
        [11083.93 , 17041.322, 17258.48 , 17303.707, 17451.951],
        [ 8211.788, 14793.737, 16515.764, 17163.238, 17235.451]],
       dtype=float32),
 array([[6041, 6073, 6040, 5998, 6060],
        [2797, 3042, 3018, 2903, 2835],
        [4969,  227,  426,  383,  249],
        [ 205,  239,  405, 4886,  231],
        [ 415, 4335, 5906, 9050, 5927]]))

# Comparison of CustomIndexPQ with faiss IndexPQ
- Slightly different neighbors but generally the same results

In [ ]:
d, m, nbits = 128, 8, 8
index_pq_compressed = faiss.IndexPQ(d, m, nbits)
index_pq_compressed.train(feature_list_compressed)
index_pq_compressed.add(feature_list_compressed)
D, I = index_pq_compressed.search(random_feature_list_compressed, 5)

In [ ]:
D[:5],I[:5]

(array([[11996.516, 19935.398, 22374.102, 22462.086, 22536.234],
        [12180.735, 20129.824, 20655.516, 20719.947, 21265.57 ],
        [11036.203, 14861.414, 15202.538, 15432.186, 16339.995],
        [ 9679.464, 17266.006, 17842.518, 18262.191, 18348.373],
        [ 8837.641, 13625.197, 15252.717, 16871.26 , 17370.398]],
       dtype=float32),
 array([[6041, 6073, 4361,  299, 4130],
        [2797, 3042, 2835, 6062, 2640],
        [4969,  383,  398,  140,  249],
        [ 205,   15,  274, 5588,  180],
        [ 415,  121, 9050, 5906, 5927]]))

In [ ]:
compare(I_flat_compressed,random_feature_list_compressed,index_pq_compressed)
compare(I_flat_compressed,random_feature_list_compressed,custom)

Recall@5 for : 0.59
Recall@5 for : 0.6


# Using more difficult VOC2012 data
- Custom code does worse than faiss 0.06 vs 0.13

In [ ]:
voc_feature_list = pickle.load(open('features/features-voc2012-resnet.pickle','rb'))
pca = PCA(n_components=128)
voc_feature_list_compressed = pca.fit_transform(voc_feature_list)

In [ ]:
random_image_indices = random.sample(range(0, len(voc_feature_list)), 100)
voc_random_feature_list = np.array([voc_feature_list[each_index] for each_index in random_image_indices])
voc_random_feature_list_compressed = np.array([voc_feature_list_compressed[each_index] for each_index in random_image_indices])

In [ ]:
#Ground truth
D_flat_compressed, I_flat_compressed = index_compressed.search(voc_random_feature_list_compressed, 6)

In [ ]:
compare(I_flat_compressed,voc_random_feature_list_compressed,index_pq_compressed)
compare(I_flat_compressed,voc_random_feature_list_compressed,custom)

Recall@5 for : 0.13
Recall@5 for : 0.06


# Custom IndexIVFPQ
- Explanation of PQ and IVFPQ at https://lear.inrialpes.fr/pubs/2011/JDS11/jegou_searching_with_quantization.pdf

**Differences between my implementation and paper**
- My inverted index did not have `{coarse_centroid: [(id1, code1),...]}` structure like in paper, but more simply `{coarse_centroid:[id1,id2,...]}` and another array of codes because this data model is easier to handle (need to learn C++, SIMD first to appreciate the storage patterns for high performance compute)

**Problems**
- Kmeans coarse clustering too slow (36 secs for 9k images, 128 dim), there must be faster way
- Search over 9k images, 128 dim takes 20 seconds, still slow


## Train and Add

### Creating coarse quantizer and residual database vectors

In [30]:
def fit_coarse_quantizer(feature_list):
    d = 128 # dimension
    m=8
    nlist = 100
    nbits = 8
    k=2**nbits

    coarse_quantizer = KMeans(n_clusters=nlist, random_state=1).fit(feature_list)
    feature_list_residual = feature_list - coarse_quantizer.cluster_centers_[coarse_quantizer.labels_]  # generate residual database vectors to be fine quantized
    return feature_list_residual, coarse_quantizer

feature_list_residual, coarse_quantizer = fit_coarse_quantizer(feature_list_compressed)

In [31]:
def apply_coarse_quantizer(feature_list):
    """Find closest IVF centroid using coarse quantizer and get residuals"""
    ivf_cells = coarse_quantizer.predict(feature_list)
    feature_list_residual = feature_list - coarse_quantizer.cluster_centers_[ivf_cells]  # generate residual database vectors to be fine quantized
    return feature_list_residual, coarse_quantizer, ivf_cells

feature_list_residual, coarse_quantizer, ivf_cells = apply_coarse_quantizer(feature_list_compressed)

In [33]:
from sklearn.cluster import KMeans

def fit_fine_quantizer(feature_list_residual, d, m, k=256):
    ds = d // m

    fine_quantizers = [KMeans(n_clusters=k, random_state=1) for _ in range(m)]

    for i in range(m):
        X_i = feature_list_residual[:, i * ds : (i + 1) * ds]
        fine_quantizers[i].fit(X_i)

    return fine_quantizers

In [36]:
fine_quantizers = fit_fine_quantizer(feature_list_residual, 128, 8)

### Product Quantizing residuals into codes

In [35]:
import numpy as np

def quantize_residuals(feature_list_residual, estimators, m):
    n, d = feature_list_residual.shape
    ds = d // m

    codes = np.empty((n, m), dtype=np.uint8)

    for i in range(m):
        estimator = estimators[i]   # ✅ FIXED
        X_i = feature_list_residual[:, i * ds : (i + 1) * ds]
        codes[:, i] = estimator.predict(X_i)

    return codes

In [37]:
fine_quantizers = fit_fine_quantizer(feature_list_residual, 128, 8)

In [38]:
m = 8
codes = quantize_residuals(feature_list_residual, fine_quantizers, m)

In [34]:
def quantize_residuals(feature_list_residual, estimators, m):
    n, d= feature_list_residual.shape
    ds = d//m
    codes = np.empty((n, m), dtype=np.uint8)  #Prevents automatic 'float64' causing IndexError: arrays used as indices must be of integer (or boolean) type

    for i in range(m):
        estimator = fine_quantizers[i]
        X_i = feature_list_residual[:, i * ds : (i + 1) * ds]
        codes[:, i] = estimator.predict(X_i)  # shape n number of vectors, m segments

    return codes

codes = quantize_residuals(feature_list_residual, fine_quantizers, m)

NameError: name 'fine_quantizers' is not defined

### Assigning residuals to inverted list

In [39]:
max_id = 0
codes_db = np.empty((0,m),dtype=np.int8) # always cleared in jupyter for convenience

In [40]:
def add_inverted_list(ivf_cells, codes, codes_db):
    from collections import defaultdict

    inverted_list = defaultdict(list)

    for idx, coarse_center in enumerate(ivf_cells):
        inverted_list[coarse_center].append(idx)

    codes_db = np.vstack([codes_db, codes])

    return inverted_list

inverted_list = add_inverted_list(ivf_cells, codes, codes_db)

# useful for checking correct number of candidates got extracted from probing during search
for key, value in inverted_list.items():
    print(key, len(value))

88 131
91 125
36 111
45 204
86 94
80 172
49 78
46 116
11 72
8 130
41 100
18 100
94 73
40 79
97 33
3 74
56 48
98 107
66 120
38 119
57 59
21 88
76 66
4 85
81 71
20 57
95 58
67 69
82 104
9 57
62 184
72 76
58 61
30 74
63 87
78 68
0 99
6 70
28 124
34 85
64 58
15 39
96 49
42 51
10 214
68 78
54 233
69 131
84 73
24 46
71 46
1 41
12 90
74 89
2 116
37 175
77 96
87 80
19 50
16 116
23 180
7 108
53 216
5 92
59 162
32 135
48 53
55 96
14 55
26 77
92 61
50 104
60 46
47 88
29 123
73 45
99 71
52 52
70 66
65 49
35 86
90 67
25 105
17 91
85 87
22 51
31 41
93 95
27 93
43 90
61 61
83 106
79 60
51 78
44 66
75 84
39 63
13 77
89 45
33 220


## Search

In [41]:
query = feature_list_compressed[[0]] #  2d to because euclidean_distances expects that
query
nprobe = 1

array([[ 3.20450630e+01,  1.24839172e+01,  8.19214344e+00,
        -2.37871342e+01,  1.08952478e-01,  2.74450207e+01,
         1.37947798e+01,  1.12661104e+01,  1.10272484e+01,
        -5.98677979e+01, -2.48901653e+00,  6.04819641e+01,
        -1.18348932e+01, -5.08927231e+01, -2.67664967e+01,
         8.51128006e+00, -8.94119167e+00, -7.20887566e+00,
        -2.86572723e+01,  1.05777254e+01,  3.08795586e+01,
        -3.06745377e+01, -6.53825474e+00, -2.15450358e+00,
         2.09681892e+00, -1.17198582e+01,  1.06501379e+01,
         1.09072676e+01, -1.37598505e+01,  1.34042053e+01,
         4.44343853e+00,  3.43707800e-01,  1.27563751e+00,
         1.44164162e+01, -8.98155880e+00, -1.89379692e+01,
        -7.72413254e+00,  7.14520073e+00, -1.70193329e+01,
         1.78796043e+01,  2.85239506e+00,  9.05057430e+00,
         1.51198826e+01,  8.60068417e+00,  5.46989918e+00,
         2.94681787e+00,  2.60889792e+00, -4.14098740e+00,
         2.44966841e+00,  2.14294884e-02,  4.17490816e+0

### Calculate distance of query vector (not residual) to all coarse centroids

In [42]:
def distance_to_IVFcentroids(query, coarse_quantizer, nprobe):
    query_distance_to_coarse_centroids = euclidean_distances(query, coarse_quantizer.cluster_centers_, squared=True)[0]
    nearest_inverted_keys = np.argsort(query_distance_to_coarse_centroids)[:nprobe]  # argsort gives index of closest coarse centroids, to be filtered by nprobe
    return nearest_inverted_keys

nearest_inverted_keys = distance_to_IVFcentroids(query, coarse_quantizer, nprobe)

### Testing calculations in 1 cell (assuming nprobe=1)

### Generating query residual vector

In [43]:
print('Closest coarse centroid: ',nearest_inverted_keys[0])

def generate_query_residual(query, coarse_quantizer, current_cell):
      # closest from coarse quantizer
    query_residual = query - coarse_quantizer.cluster_centers_[current_cell]  # generate residual query to be compared against all quantized residuals
    # print(query_residual.shape, query_residual[:3])

    return query_residual

current_cell = nearest_inverted_keys[0]
query_residual = generate_query_residual(query, coarse_quantizer, current_cell)


Closest coarse centroid:  88


### Generating distance table

In [45]:
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances

def compute_distance_table(query_residual, fine_quantizers, m=8, k=256):

    distance_table = np.empty((m, k), dtype=np.float32)

    d = query_residual.shape[1]
    ds = d // m

    for i in range(m):
        centers = fine_quantizers[i].cluster_centers_
        X_i = query_residual[:, i * ds : (i + 1) * ds]

        distance_table[i, :] = euclidean_distances(X_i, centers, squared=True)

    return distance_table

In [46]:
distance_table = compute_distance_table(query_residual, fine_quantizers)

In [44]:
def compute_distance_table(query_residual, fine_quantizers):
    distance_table = np.empty((m, k), dtype=np.float32)  # shape m segments, distance to k clusters

    d = query_residual.shape[1]
    ds = d//m
    for i in range(m):
        X_i = query_residual[:, i * ds : (i + 1) * ds]
        centers = fine_quantizers[i].cluster_centers_  # (k, ds)
        distance_table[i, :] = euclidean_distances(X_i, centers, squared=True)
    return distance_table
distance_table = compute_distance_table(query_residual, fine_quantizers)

NameError: name 'k' is not defined

### Filtering residual vectors using inverted list

In [47]:
def filter_residual_vectors(inverted_list, codes, current_cell):

    filtered_ids = inverted_list[current_cell]
    filtered_result = codes[filtered_ids]
    # print(filtered_result.shape, filtered_result[:3])

    return filtered_result, filtered_ids

current_cell = nearest_inverted_keys[0]

filtered_result, filtered_ids = filter_residual_vectors(inverted_list, codes, current_cell)

### Calculating distances on filtered residual vectors

In [48]:
def calculate_distances(filtered_result, distance_table):
    distances = np.zeros(len(filtered_result), dtype=np.float32)

    for i in range(m):
        distances += distance_table[i, filtered_result[:, i]]
    # print(distances.shape, distances[:3])

    return distances

distances = calculate_distances(filtered_result, distance_table)


In [49]:
def find_smallest_k(distances, filtered_ids, k_nearest):
    import heapq
    import operator

    distance_id = zip(distances,filtered_ids)
    D, I = zip(*heapq.nsmallest(k_nearest, distance_id, operator.itemgetter(0)))

    return D, I

k_nearest = 6
D, I = find_smallest_k(distances, filtered_ids, k_nearest)   # generators can only be used once! Re-run previous cell to regenerate distance_id if needed
D, I

((np.float32(9370.197),
  np.float32(15346.791),
  np.float32(16168.11),
  np.float32(16893.924),
  np.float32(17912.2),
  np.float32(18029.076)),
 (0, 168, 8, 25, 229, 177))

### Repeat above steps for all query vectors, and all nprobe

In [50]:
def search(query,
                  coarse_quantizer,
                  nprobe,
                  codes,
                  k_nearest
                  ):

    nearest_inverted_keys = distance_to_IVFcentroids(query, coarse_quantizer, nprobe)

    nprobe_distances = np.array([], dtype=np.float32)
    nprobe_filtered_ids = np.array([], dtype=np.uint64)

    for current_cell in nearest_inverted_keys:
        query_residual = generate_query_residual(query, coarse_quantizer, current_cell)
        distance_table = compute_distance_table(query_residual, fine_quantizers)
        filtered_result, filtered_ids = filter_residual_vectors(inverted_list, codes, current_cell)
        distances = calculate_distances(filtered_result, distance_table)
        nprobe_distances = np.append(nprobe_distances, distances)
        nprobe_filtered_ids = np.append(nprobe_filtered_ids, filtered_ids)

    D, I = find_smallest_k(nprobe_distances, nprobe_filtered_ids, k_nearest)

    return D, I

## Testing how recall varies with nprobe

In [51]:
k_nearest = 6
m, k = 8, 2**nbits

nprobe_test = {}
# nprobes_to_test = range(1,4)
nprobes_to_test = [1]

for nprobe in nprobes_to_test:
    D_list = []
    I_list = []
    for query in feature_list_compressed:
        D, I = search(query.reshape(1,-1),   #sklearn euclidean_distances needs 2D
                            coarse_quantizer,
                            nprobe,
                            codes,
                            k_nearest)
        D_list.append(D)
        I_list.append(I)

    D_list = np.array(D_list, dtype=np.float32)
    I_list = np.array(I_list, dtype=np.uint64)

    nprobe_test[nprobe] = (D_list, I_list)

In [53]:
I_list_subset = I_list[:100]

In [54]:
print(
    f'Recall@5:',
    (I_list_subset[:, 1:6] == I_flat[:, [1]]).sum() / len(I_flat)
)

Recall@5: 0.01


In [55]:
recall = 0

for i in range(len(I_flat)):
    if I_flat[i, 1] in I_list[i, 1:6]:
        recall += 1

recall /= len(I_flat)

print(f"Recall@5: {recall}")

Recall@5: 0.01


In [52]:
I_list = nprobe_test[1][1]
print(f'Recall@5 for Custom IndexIVFPQ nprobe={nprobe}:', (I_list[:,1:6] == I_flat[:,[1]]).sum()/len(feature_list_compressed))

ValueError: operands could not be broadcast together with shapes (9144,5) (100,1) 

## Comparing Custom IVFPQ with faiss

In [56]:
import faiss

d = 128 # dimension
nlist = 100
m=8
nbits = 8

quantizer = faiss.IndexFlatL2(d)
index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)

index.train(feature_list_compressed)
index.add(feature_list_compressed)

In [57]:
k_nearest = 5
D, I = index.search(feature_list_compressed, k_nearest)
D[:5], I[:5]

(array([[ 8390.028 , 16984.994 , 17739.887 , 19010.252 , 19052.488 ],
        [ 9440.412 , 18739.865 , 20239.092 , 20720.752 , 20882.79  ],
        [ 2693.544 ,  4015.2476,  4207.3413,  4213.7026,  4215.1475],
        [ 6730.733 , 14164.1455, 14308.248 , 15067.272 , 15217.457 ],
        [11475.555 , 22474.715 , 23120.346 , 23389.242 , 23536.002 ]],
       dtype=float32),
 array([[   0,    8,  357,  364,  405],
        [   1,   76,  213,   54,  303],
        [   2,  446,  306,  431,  157],
        [   3,  208, 6222,  393,  273],
        [   4, 2448, 6020, 2496, 6067]]))

In [58]:
index_flat = faiss.IndexFlatL2(d)
index_flat.add(feature_list_compressed)
D_flat, I_flat = index_flat.search(feature_list_compressed, k_nearest)
D_flat[:5], I_flat[:5]

(array([[0.0000000e+00, 1.6330896e+04, 1.7000059e+04, 2.2679613e+04,
         2.3235717e+04],
        [1.5625000e-02, 2.1283279e+04, 2.4666084e+04, 2.4895584e+04,
         2.4959203e+04],
        [2.3437500e-02, 3.4765625e+03, 3.7983711e+03, 3.9878984e+03,
         4.5920938e+03],
        [0.0000000e+00, 1.2589309e+04, 1.3675467e+04, 1.4152389e+04,
         1.4636859e+04],
        [3.9062500e-03, 2.2252309e+04, 2.3779234e+04, 2.3815738e+04,
         2.4989002e+04]], dtype=float32),
 array([[   0,    8,  168, 7939, 5586],
        [   1,  149,  344,  213,  309],
        [   2,  138,  431,  157,  193],
        [   3,  426,   75,  208, 6222],
        [   4, 5403, 6021, 2562, 2533]]))

In [59]:
compare(I_flat, feature_list_compressed, index, 'IndexIVFPQ (compressed)')
for nprobe, (D_list, I_list) in nprobe_test.items():
    print(f'Recall@5 for Custom IndexIVFPQ nprobe={nprobe}:', (I_list[:,1:6] == I_flat[:,[1]]).sum()/len(feature_list_compressed))

Recall@5 for IndexIVFPQ (compressed): 0.6644794400699913
Recall@5 for Custom IndexIVFPQ nprobe=1: 0.6823053368328958


### Implementation correctness

- Custom IVFPQ implementation looks right based on high Recall@5 (even higher than faiss)
- Increasing nprobe from 1 to 2 to 3 shows increasing number of true nearest neighbors found in top 5 neighbors (excluding self)
- Time to train and search is slow though

## Packing functions into a class to simplify method signatures

In [60]:
from collections import defaultdict
import heapq
import operator

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances

class CustomIndexIVFPQ:

    BITS2DTYPE = {
        8: np.uint8,
        16: np.uint16,
    }

    def __init__(self,
                 d: int,
                 m: int,
                 nlist: int,
                 nbits: int,) -> None:
        """Custom IndexIVFPQ implementation.

        Parameters
        ----------
        d
            Dimensionality of the original vectors.
        m
            Number of segments.
        nlist
            Number of coarse centroids for IVF
        nbits
            Number of bits.
        """
        if d % m != 0:
            raise ValueError("d needs to be a multiple of m")

        if nbits not in CustomIndexIVFPQ.BITS2DTYPE:
            raise ValueError(f"Unsupported number of bits {nbits}")

        self.m = m
        self.nlist = nlist
        self.nprobe = 1
        self.k = 2**nbits
        self.d = d
        self.ds = d // m

        self.coarse_quantizer = KMeans(n_clusters=self.nlist, random_state=1)
        self.inverted_list = defaultdict(list)
        self.max_id = 0 # to start following batches of vector adding from the right index to prevent duplicate ids if in different IVF cell or overwriting of data if in same IVF cell
        self.codes_db = np.empty((0,m),dtype=CustomIndexIVFPQ.BITS2DTYPE[nbits]) # always cleared in jupyter for convenience

        self.fine_quantizers = [KMeans(n_clusters=self.k, random_state=1) for _ in range(m)]

        self.is_trained = False

        self.dtype = CustomIndexIVFPQ.BITS2DTYPE[nbits]
        self.dtype_orig = np.float32

    def fit_coarse_quantizer(self, feature_list):
        """Coarse quantizer to divide database vectors into various voronoi cells so search only probes nprobe closest

        Parameters
        ----------
        feature_list
            Data to be converted to residuals

        Returns
        -------
        feature_list_residual
            Residuals created by subtracting each vector with it's own coarse centroid
        """
        self.coarse_quantizer.fit(feature_list)
        feature_list_residual = feature_list - self.coarse_quantizer.cluster_centers_[self.coarse_quantizer.labels_]  # generate residual database vectors to be fine quantized
        return feature_list_residual

    def fit_fine_quantizer(self, feature_list_residual):
        """Fit m fine quantizers for each of m segments

        Parameters
        ----------
        feature_list_residual
            Residuals created by subtracting each vector with it's own coarse centroid
        """
        for i in range(self.m):
            X_i = feature_list_residual[:, i * self.ds : (i + 1) * self.ds]
            self.fine_quantizers[i].fit(X_i)

    def apply_coarse_quantizer(self, feature_list):
        """Find closest IVF centroid using coarse quantizer and get residuals

        Parameters
        ----------
        feature_list
            Raw Vectors to be assigned to coarse quantizer centroids

        Returns
        -------
        feature_list_residual
            Residuals to be quantized by fine quantizer after coarse quantization

        ivf_cells
            labels of which coarse centroids each vector goes into
        """
        ivf_cells = self.coarse_quantizer.predict(feature_list)
        feature_list_residual = feature_list - self.coarse_quantizer.cluster_centers_[ivf_cells]  # generate residual database vectors to be fine quantized
        return feature_list_residual, ivf_cells

    def quantize_residuals(self, feature_list_residual):
        """
        Fine quantization of residuals of both database and query vectors

        Parameters
        ----------
        feature_list_residual
            Residuals created by subtracting each vector with it's own coarse centroid

        Returns
        -------
        codes
            Quantized codes of residuals, shaped n, m
        """
        n = len(feature_list_residual)
        codes = np.empty((n, self.m), dtype=self.dtype)  #Prevents automatic 'float64' causing IndexError: arrays used as indices must be of integer (or boolean) type

        for i in range(self.m):
            estimator = self.fine_quantizers[i]
            X_i = feature_list_residual[:, i * self.ds : (i + 1) * self.ds]
            codes[:, i] = estimator.predict(X_i)  # shape n number of vectors, m segments

        return codes

    def add_inverted_list(self, ivf_cells, codes):
        """
        Assign ids to cells and add codes to database

        Parameters
        ----------
        ivf_cells
            coarse quantized labels of vectors
        codes
            Quantized codes of residuals to be added to database of quantized vectors
        """
        for idx, coarse_center in enumerate(ivf_cells, start=self.max_id):
            self.inverted_list[coarse_center].append(idx)

        self.max_id += len(codes) # update max_id so next addition to IVF don't duplicate id (if same different coarse_center) or overwrite data (if same coarse_center)
        self.codes_db = np.vstack([self.codes_db, codes])

    def distance_to_IVFcentroids(self, query):
        """
        Find distance of raw query to coarse centroids

        Parameters
        ----------
        query
            Raw query vector

        """
        query_distance_to_coarse_centroids = euclidean_distances(query, self.coarse_quantizer.cluster_centers_, squared=True)[0]
        nearest_inverted_keys = np.argsort(query_distance_to_coarse_centroids)[:self.nprobe]  # argsort gives index of closest coarse centroids, to be filtered by nprobe
        return nearest_inverted_keys

    def generate_query_residual(self, query, current_cell):
        """Find closest IVF centroid and return residual of query from that centroid

        Parameters
        ----------
        query
            Raw query vector
        current_cell
            A particular coarse centroid explored during probing for IVF cells

        Returns
        -------
        """
        query_residual = query - self.coarse_quantizer.cluster_centers_[current_cell]  # generate residual query to be compared against all quantized residuals

        return query_residual

    def compute_distance_table(self, query_residual):
        """Distance table per coarse centroid for reuse by all quantized residual vectors in same cell

        Parameters
        ----------
        query_residual
            Residual of query vector

        Returns
        -------
        distance_table
            Table of distances from each query vector to all k clusters, for each segment, to be used by all database vectors in same coarse centroid
        """
        distance_table = np.empty((self.m, self.k), dtype=self.dtype_orig)  # shape m segments, distance to k clusters

        for i in range(self.m):
            X_i = query_residual[:, i * self.ds : (i + 1) * self.ds]
            centers = self.fine_quantizers[i].cluster_centers_  # (k, ds)
            distance_table[i, :] = euclidean_distances(X_i, centers, squared=True)
        return distance_table

    def filter_residual_vectors(self, current_cell):
        """Identify only relevant vectors in same cell as query to compute distances for

        Parameters
        ----------
        current_cell
            particular coarse centroid explored during probing for IVF cells

        Returns
        -------
        filtered_result
            Filtered codes so search only calculates their distances
        filtered_ids
            Filtered ids used for indicating which vectors are being searched
        """

        filtered_ids = self.inverted_list[current_cell]
        filtered_result = self.codes_db[filtered_ids]

        return filtered_result, filtered_ids

    def calculate_distances(self, filtered_result, distance_table):
        """Calculate distance of each database vector with quantized query vector

        Parameters
        ----------
        filtered_result
            Filtered codes for calculating distances with their coarse centroid
        distance_table
            Table of distances from each query vector to all k clusters, for each segment, to be used by all database vectors in same coarse centroid

        Returns
        -------
        distances
            Distance of each database vector with quantized query vector
        """
        distances = np.zeros(len(filtered_result), dtype=self.dtype_orig)

        for i in range(m):
            distances += distance_table[i, filtered_result[:, i]]

        return distances

    def find_smallest_k(self, distances, filtered_ids, k_nearest):
        """Find nearest k neighbors (including self)

        Parameters
        ----------
        distances
            Distance of each database vector with quantized query vector
        filtered_ids
            ids of vectors
        k_nearest
            Number of approximate neighbors


        Returns
        -------
        D
            K nearest distances per query vector
        I
            Indices of the K nearest distances per query vector
        """
        distance_id = zip(distances,filtered_ids)
        D, I = zip(*heapq.nsmallest(k_nearest, distance_id, operator.itemgetter(0)))

        return D, I

    def train(self, feature_list: np.ndarray) -> None:
        """Train the index given data

        Parameters
        ----------
        feature_list
            Array of shape `(n, d)` and dtype `float32`.
        """
        feature_list_residual = self.fit_coarse_quantizer(feature_list)
        self.fit_fine_quantizer(feature_list_residual)

        self.is_trained = True

    def add(self, feature_list: np.ndarray) -> None:
        """Add vectors to the database (their encoded versions).

        Parameters
        ----------
        feature_list
            Array of shape `(n_codes, d)` of dtype `np.float32`.
        Raises
        ------
        ValueError
            Cannot add data if quantizers not trained
        """
        if not self.is_trained:
            raise ValueError("Both coarse and fine quantizers need to be trained first.")

        feature_list_residual, ivf_cells = self.apply_coarse_quantizer(feature_list)
        codes = self.quantize_residuals(feature_list_residual)
        self.add_inverted_list(ivf_cells, codes)

    def search(self, query: np.ndarray, k_nearest: int) -> tuple:
        """Search for k nearest neighbors

        Parameters
        ----------
        query
            Raw query vector
        k_nearest
            Number of approximate neighbors


        Returns
        -------
        D
            K nearest distances per query vector
        I
            Indices of the K nearest distances per query vector

        Raises
        ------
        ValueError
            Cannot add data if quantizers not trained
            Cannot search database if it's empty

        """
        if not self.is_trained:
            raise ValueError("Both coarse and fine quantizers need to be trained first.")

        if self.codes_db.size == 0:
            raise ValueError("No codes detected. You need to run `add` first")

        nearest_inverted_keys = self.distance_to_IVFcentroids(query)

        nprobe_distances = np.array([], dtype=self.dtype_orig)
        nprobe_filtered_ids = np.array([], dtype=np.uint64)

        for current_cell in nearest_inverted_keys:
            query_residual = self.generate_query_residual(query, current_cell)
            distance_table = self.compute_distance_table(query_residual)
            filtered_result, filtered_ids = self.filter_residual_vectors(current_cell)
            distances = self.calculate_distances(filtered_result, distance_table)

            nprobe_distances = np.append(nprobe_distances, distances)
            nprobe_filtered_ids = np.append(nprobe_filtered_ids, filtered_ids)

        D, I = self.find_smallest_k(nprobe_distances, nprobe_filtered_ids, k_nearest)

        # some will return < k neighbors, need to pad on right to form rectangular result array
        # assigning -1 into uint8 causes 18446744073709551615, ok for evaluation as long as doesn't match a ground truth index
        return np.pad(D,pad_width=(0,k_nearest-len(D)),constant_values=-1), np.pad(I,pad_width=(0,k_nearest-len(I)),constant_values=-1)


In [61]:
d = 128
m = 8
nlist = 100
nbits = 8
custom_ivfpq = CustomIndexIVFPQ(d,m,nlist,nbits)

custom_ivfpq.train(feature_list_compressed)

In [62]:
custom_ivfpq.inverted_list
custom_ivfpq.codes_db

defaultdict(list, {})

array([], shape=(0, 8), dtype=uint8)

In [63]:
custom_ivfpq.add(feature_list_compressed)

In [64]:
k_nearest = 6

nprobe_test = {}
# nprobes_to_test = range(1,4)
nprobes_to_test = [1]

for nprobe in nprobes_to_test:
    D_list = []
    I_list = []
    for query in feature_list_compressed:
        D, I = custom_ivfpq.search(query.reshape(1,-1),   #sklearn euclidean_distances needs 2D
                                   k_nearest)
        D_list.append(D)
        I_list.append(I)

    D_list = np.array(D_list, dtype=np.float32)
    I_list = np.array(I_list, dtype=np.uint64)

    nprobe_test[nprobe] = (D_list, I_list)

In [65]:
I_list = nprobe_test[1][1]
print(f'Recall@5 for Custom IndexIVFPQ nprobe={nprobe}:', (I_list[:,1:6] == I_flat[:,[1]]).sum()/len(feature_list_compressed))

Recall@5 for Custom IndexIVFPQ nprobe=1: 0.6823053368328958


# Refactoring Custom IndexIVFPQ to switch both quantizers from sklearn to faiss.Kmeans
- Kmeans is too slow to train (40 secs on 9k caltech)

**Identifying API Changes** (preparing for find and replace)

|      Sklearn      |  Faiss |
|:-----------------:|-------:|
|   model.fit(x) | model.train(x) |
|   model.predict(x)|model.assign(x)[1] |
|   model.labels_|   model.assign(x)[1] |
|   model.cluster_centers_|  model.centroids |

## Proving shape equivalence between sklearn and faiss

In [66]:
import faiss
from sklearn.cluster import KMeans

np.random.seed(1)
x = np.random.random((10000,100))
ncentroids = 128
niter = 20
verbose = True
d = x.shape[1]

model = KMeans(n_clusters=ncentroids, random_state=1)
model.fit(x)
print('Kmeans predict shape: ', model.predict(x).shape)
print('Kmeans labels shape: ', model.labels_.shape)
print('Kmeans cluster centers shape: ', model.cluster_centers_.shape)

kmeans = faiss.Kmeans(d, ncentroids, verbose=verbose, seed=1)
kmeans.train(x) # model.fit()

print('faiss predict shape: ', kmeans.assign(x)[1].shape) # model.labels_ or model.predict(x)
print('faiss cluster centers shape ', kmeans.centroids.shape) # model.cluster_centers_



KMeans(n_clusters=128, random_state=1)

Kmeans predict shape:  (10000,)
Kmeans labels shape:  (10000,)
Kmeans cluster centers shape:  (128, 100)


np.float64(74839.5)

faiss predict shape:  (10000,)
faiss cluster centers shape  (128, 100)


In [67]:
from collections import defaultdict
import heapq
import operator

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances
import faiss


class CustomIndexIVFPQ:
    BITS2DTYPE = {
        8: np.uint8,
        16: np.uint16,
    }

    def __init__(
        self,
        d: int,
        m: int,
        nlist: int,
        nbits: int,
    ) -> None:
        """Custom IndexIVFPQ implementation.

        Parameters
        ----------
        d
            Dimensionality of the original vectors.
        m
            Number of segments.
        nlist
            Number of coarse centroids for IVF
        nbits
            Number of bits.
        """
        if d % m != 0:
            raise ValueError("d needs to be a multiple of m")

        if nbits not in CustomIndexIVFPQ.BITS2DTYPE:
            raise ValueError(f"Unsupported number of bits {nbits}")

        self.m = m
        self.nlist = nlist
        self.nprobe = 1
        self.k = 2**nbits
        self.d = d
        self.ds = d // m

        # self.coarse_quantizer = KMeans(n_clusters=self.nlist, random_state=1)
        self.coarse_quantizer = faiss.Kmeans(d, nlist, seed=1)
        self.inverted_list = defaultdict(list)
        self.max_id = 0  # to start following batches of vector adding from the right index to prevent duplicate ids if in different IVF cell or overwriting of data if in same IVF cell
        self.codes_db = np.empty(
            (0, m), dtype=CustomIndexIVFPQ.BITS2DTYPE[nbits]
        )

        self.fine_quantizers = [
            # KMeans(n_clusters=self.k, random_state=1) for _ in range(m)
            faiss.Kmeans(self.ds, self.k, seed=1) for _ in range(m)
        ]

        self.is_trained = False

        self.dtype = CustomIndexIVFPQ.BITS2DTYPE[nbits]
        self.dtype_orig = np.float32

    def fit_coarse_quantizer(self, feature_list):
        """Coarse quantizer to divide database vectors into various voronoi cells so search only probes nprobe closest

        Parameters
        ----------
        feature_list
            Data to be converted to residuals

        Returns
        -------
        feature_list_residual
            Residuals created by subtracting each vector with it's own coarse centroid
        """
        self.coarse_quantizer.train(feature_list)
        feature_list_residual = (
            feature_list
            - self.coarse_quantizer.centroids[self.coarse_quantizer.assign(feature_list)[1]]
        )  # generate residual database vectors to be fine quantized
        return feature_list_residual

    def fit_fine_quantizer(self, feature_list_residual):
        """Fit m fine quantizers for each of m segments

        Parameters
        ----------
        feature_list_residual
            Residuals created by subtracting each vector with it's own coarse centroid
        """
        for i in range(self.m):
            X_i = feature_list_residual[:, i * self.ds : (i + 1) * self.ds]
            self.fine_quantizers[i].train(X_i)

    def apply_coarse_quantizer(self, feature_list):
        """Find closest IVF centroid using coarse quantizer and get residuals

        Parameters
        ----------
        feature_list
            Raw Vectors to be assigned to coarse quantizer centroids

        Returns
        -------
        feature_list_residual
            Residuals to be quantized by fine quantizer after coarse quantization

        ivf_cells
            labels of which coarse centroids each vector goes into
        """
        ivf_cells = self.coarse_quantizer.assign(feature_list)[1]
        feature_list_residual = (
            feature_list - self.coarse_quantizer.centroids[ivf_cells]
        )  # generate residual database vectors to be fine quantized
        return feature_list_residual, ivf_cells

    def quantize_residuals(self, feature_list_residual):
        """
        Fine quantization of residuals of both database and query vectors

        Parameters
        ----------
        feature_list_residual
            Residuals created by subtracting each vector with it's own coarse centroid

        Returns
        -------
        codes
            Quantized codes of residuals, shaped n, m
        """
        n = len(feature_list_residual)
        codes = np.empty(
            (n, self.m), dtype=self.dtype
        )  # Prevents automatic 'float64' causing IndexError: arrays used as indices must be of integer (or boolean) type

        for i in range(self.m):
            estimator = self.fine_quantizers[i]
            X_i = feature_list_residual[:, i * self.ds : (i + 1) * self.ds]
            codes[:, i] = estimator.assign(
                X_i)[1]  # shape n number of vectors, m segments

        return codes

    def add_inverted_list(self, ivf_cells, codes):
        """
        Assign ids to cells and add codes to database

        Parameters
        ----------
        ivf_cells
            coarse quantized labels of vectors
        codes
            Quantized codes of residuals to be added to database of quantized vectors
        """
        for idx, coarse_center in enumerate(ivf_cells, start=self.max_id):
            self.inverted_list[coarse_center].append(idx)

        self.max_id += len(
            codes
        )  # update max_id so next addition to IVF don't duplicate id (if same different coarse_center) or overwrite data (if same coarse_center)
        self.codes_db = np.vstack([self.codes_db, codes])

    def distance_to_IVFcentroids(self, query):
        """
        Find distance of raw query to coarse centroids

        Parameters
        ----------
        query
            Raw query vector

        """
        query_distance_to_coarse_centroids = euclidean_distances(
            query, self.coarse_quantizer.centroids, squared=True
        )[0]
        nearest_inverted_keys = np.argsort(query_distance_to_coarse_centroids)[
            : self.nprobe
        ]  # argsort gives index of closest coarse centroids, to be filtered by nprobe
        return nearest_inverted_keys

    def generate_query_residual(self, query, current_cell):
        """Find closest IVF centroid and return residual of query from that centroid

        Parameters
        ----------
        query
            Raw query vector
        current_cell
            A particular coarse centroid explored during probing for IVF cells

        Returns
        -------
        """
        query_residual = (
            query - self.coarse_quantizer.centroids[current_cell]
        )  # generate residual query to be compared against all quantized residuals

        return query_residual

    def compute_distance_table(self, query_residual):
        """Distance table per coarse centroid for reuse by all quantized residual vectors in same cell

        Parameters
        ----------
        query_residual
            Residual of query vector

        Returns
        -------
        distance_table
            Table of distances from each query vector to all k clusters, for each segment, to be used by all database vectors in same coarse centroid
        """
        distance_table = np.empty(
            (self.m, self.k), dtype=self.dtype_orig
        )  # shape m segments, distance to k clusters

        for i in range(self.m):
            X_i = query_residual[:, i * self.ds : (i + 1) * self.ds]
            centers = self.fine_quantizers[i].centroids  # (k, ds)
            distance_table[i, :] = euclidean_distances(X_i, centers, squared=True)
        return distance_table

    def filter_residual_vectors(self, current_cell):
        """Identify only relevant vectors in same cell as query to compute distances for

        Parameters
        ----------
        current_cell
            particular coarse centroid explored during probing for IVF cells

        Returns
        -------
        filtered_result
            Filtered codes so search only calculates their distances
        filtered_ids
            Filtered ids used for indicating which vectors are being searched
        """

        filtered_ids = self.inverted_list[current_cell]
        filtered_result = self.codes_db[filtered_ids]

        return filtered_result, filtered_ids

    def calculate_distances(self, filtered_result, distance_table):
        """Calculate distance of each database vector with quantized query vector

        Parameters
        ----------
        filtered_result
            Filtered codes for calculating distances with their coarse centroid
        distance_table
            Table of distances from each query vector to all k clusters, for each segment, to be used by all database vectors in same coarse centroid

        Returns
        -------
        distances
            Distance of each database vector with quantized query vector
        """
        distances = np.zeros(len(filtered_result), dtype=self.dtype_orig)

        for i in range(m):
            distances += distance_table[i, filtered_result[:, i]]

        return distances

    def find_smallest_k(self, distances, filtered_ids, k_nearest):
        """Find nearest k neighbors (including self)

        Parameters
        ----------
        distances
            Distance of each database vector with quantized query vector
        filtered_ids
            ids of vectors
        k_nearest
            Number of approximate neighbors


        Returns
        -------
        D
            K nearest distances per query vector
        I
            Indices of the K nearest distances per query vector
        """
        distance_id = zip(distances, filtered_ids)
        D, I = zip(*heapq.nsmallest(k_nearest, distance_id, operator.itemgetter(0)))

        return D, I

    def train(self, feature_list: np.ndarray) -> None:
        """Train the index given data

        Parameters
        ----------
        feature_list
            Array of shape `(n, d)` and dtype `float32`.
        """
        feature_list_residual = self.fit_coarse_quantizer(feature_list)
        self.fit_fine_quantizer(feature_list_residual)

        self.is_trained = True

    def add(self, feature_list: np.ndarray) -> None:
        """Add vectors to the database (their encoded versions).

        Parameters
        ----------
        feature_list
            Array of shape `(n_codes, d)` of dtype `np.float32`.
        Raises
        ------
        ValueError
            Cannot add data if quantizers not trained
        """
        if not self.is_trained:
            raise ValueError(
                "Both coarse and fine quantizers need to be trained first."
            )

        feature_list_residual, ivf_cells = self.apply_coarse_quantizer(feature_list)
        codes = self.quantize_residuals(feature_list_residual)
        self.add_inverted_list(ivf_cells, codes)

    def search(self, query: np.ndarray, k_nearest: int) -> tuple:
        """Search for k nearest neighbors

        Parameters
        ----------
        query
            Raw query vector
        k_nearest
            Number of approximate neighbors


        Returns
        -------
        D
            K nearest distances per query vector
        I
            Indices of the K nearest distances per query vector

        Raises
        ------
        ValueError
            Cannot add data if quantizers not trained
            Cannot search database if it's empty

        """
        if not self.is_trained:
            raise ValueError(
                "Both coarse and fine quantizers need to be trained first."
            )

        if self.codes_db.size == 0:
            raise ValueError("No codes detected. You need to run `add` first")

        nearest_inverted_keys = self.distance_to_IVFcentroids(query)

        nprobe_distances = np.array([], dtype=self.dtype_orig)
        nprobe_filtered_ids = np.array([], dtype=np.uint64)

        for current_cell in nearest_inverted_keys:
            query_residual = self.generate_query_residual(query, current_cell)
            distance_table = self.compute_distance_table(query_residual)
            filtered_result, filtered_ids = self.filter_residual_vectors(current_cell)
            distances = self.calculate_distances(filtered_result, distance_table)

            nprobe_distances = np.append(nprobe_distances, distances)
            nprobe_filtered_ids = np.append(nprobe_filtered_ids, filtered_ids)

        D, I = self.find_smallest_k(nprobe_distances, nprobe_filtered_ids, k_nearest)

        # some will return < k neighbors, need to pad on right to form rectangular result array
        # assigning -1 into uint8 causes 18446744073709551615, ok for evaluation as long as doesn't match a ground truth index
        return np.pad(D,pad_width=(0,k_nearest-len(D)),constant_values=-1), np.pad(I,pad_width=(0,k_nearest-len(I)),constant_values=-1)


## Verifying correctness of refactoring
- Recall@5 for coarse quantizer using faiss instead of sklean kmeans dropped to 0.66 from 0.69
- Training speed improved from 40 seconds to < 2seconds

In [68]:
d = 128
m = 8
nlist = 100
nbits = 8
custom_ivfpq = CustomIndexIVFPQ(d,m,nlist,nbits)

custom_ivfpq.train(feature_list_compressed)

In [69]:
custom_ivfpq.add(feature_list_compressed)

In [70]:
k_nearest = 6

nprobe_test = {}
# nprobes_to_test = range(1,4)
nprobes_to_test = [1]

for nprobe in nprobes_to_test:
    D_list = []
    I_list = []

    for query in feature_list_compressed:
        D, I = custom_ivfpq.search(query.reshape(1,-1),   #sklearn euclidean_distances needs 2D
                                k_nearest)
        D_list.append(D)
        I_list.append(I)

    D_list = np.array(D_list, dtype=np.float32)
    I_list = np.array(I_list, dtype=np.uint64)

    nprobe_test[nprobe] = (D_list, I_list)

In [71]:
I_list = nprobe_test[1][1]
print(f'Recall@5 for Custom IndexIVFPQ nprobe={nprobe}:', (I_list[:,1:6] == I_flat[:,[1]]).sum()/len(feature_list_compressed))

Recall@5 for Custom IndexIVFPQ nprobe=1: 0.6696194225721784


In [72]:
import pickle
import os

SAVE_PATH = "/content/drive/MyDrive/ahrefs/features"
os.makedirs(SAVE_PATH, exist_ok=True)

# Save everything
pickle.dump(feature_list, open(SAVE_PATH + '/features-caltech101-resnet.pickle','wb'))
pickle.dump(feature_list_compressed, open(SAVE_PATH + '/pca_features.pickle','wb'))
pickle.dump(feature_list_residual, open(SAVE_PATH + '/residual_features.pickle','wb'))
pickle.dump(fine_quantizers, open(SAVE_PATH + '/fine_quantizers.pickle','wb'))
pickle.dump(codes, open(SAVE_PATH + '/codes.pickle','wb'))